# OBAD L1 Candidate C
Note dùng để fine turn lại C1 sau khi kết quả train 2 lớp chưa dủ tốt


In [1]:
#cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#cell 3: khai báo
from pathlib import Path
import subprocess, sys, json, os, platform, psutil, torch
PROJECT_ROOT=Path('/content/drive/MyDrive/OBAD')
RUN_ID='l1_candidate_c_current'
SNAPSHOT=PROJECT_ROOT/'data/dataModel/l1_adaptation/source_snapshots/l1_candidate_c_source_current'
ADAPTATION=PROJECT_ROOT/'data/realtime_audit/l1_adaptation_eval_20260715_162915'
PACKAGE=PROJECT_ROOT/'data/dataModel/l1_adaptation'/RUN_ID
def run_checked(args):
    return subprocess.run(args, cwd=PROJECT_ROOT, text=True, capture_output=False, check=True)
assert SNAPSHOT.exists() and ADAPTATION.exists()

In [3]:
#cell 4
run_checked([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    "requirements2.txt",
    "pyarrow",
    "pyyaml",
])

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "ram_gb": round(
        psutil.virtual_memory().total / 2**30,
        1,
    ),
    "drive_free_gb": round(
        os.statvfs(PROJECT_ROOT).f_bavail
        * os.statvfs(PROJECT_ROOT).f_frsize
        / 2**30,
        1,
    ),
})

assert torch.cuda.is_available()

{'python': '3.12.13', 'torch': '2.11.0+cu128', 'cuda': True, 'gpu': 'Tesla T4', 'ram_gb': 12.7, 'drive_free_gb': 62.6}


In [4]:
#cell 5: chỉ chạy lần đầu
run_checked([
    sys.executable,
    "modeling/l1_tcn/scripts/run_candidate_c_colab.py",
    "validate-source",
    "--source-snapshot-dir",
    str(SNAPSHOT),
])

CompletedProcess(args=['/usr/bin/python3', 'modeling/l1_tcn/scripts/run_candidate_c_colab.py', 'validate-source', '--source-snapshot-dir', '/content/drive/MyDrive/OBAD/data/dataModel/l1_adaptation/source_snapshots/l1_candidate_c_source_current'], returncode=0)

In [5]:
#cell 6: chạy 1 lần
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/OBAD")

RUN_ID = "l1_candidate_c_current"

SNAPSHOT = (
    PROJECT_ROOT
    / "data/dataModel/l1_adaptation/source_snapshots"
    / "l1_candidate_c_source_current"
)

ADAPTATION = (
    PROJECT_ROOT
    / "data/realtime_audit"
    / "l1_adaptation_eval_20260715_162915"
)

PACKAGE = (
    PROJECT_ROOT
    / "data/dataModel/l1_adaptation"
    / RUN_ID
)

assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert SNAPSHOT.exists(), SNAPSHOT
assert ADAPTATION.exists(), ADAPTATION

command = [
    sys.executable,
    "-u",
    "modeling/l1_tcn/scripts/run_candidate_c_colab.py",
    "prepare",
    "--source-mode",
    "snapshot",
    "--source-snapshot-dir",
    str(SNAPSHOT),
    "--adaptation-audit-dir",
    str(ADAPTATION),
    "--candidate-run-id",
    RUN_ID,
    "--candidate-package-dir",
    str(PACKAGE),
    "--resume",
]

environment = os.environ.copy()
environment["PYTHONUNBUFFERED"] = "1"
environment["PYTHONIOENCODING"] = "utf-8"

print("=== RESUME CANDIDATE C PREPARE ===", flush=True)
print(" ".join(map(str, command)), flush=True)

subprocess.run(
    command,
    cwd=PROJECT_ROOT,
    env=environment,
    text=True,
    check=True,
)

print(
    "\nL1 CANDIDATE C PREPARATION FINISHED SUCCESSFULLY",
    flush=True,
)

=== RESUME CANDIDATE C PREPARE ===
/usr/bin/python3 -u modeling/l1_tcn/scripts/run_candidate_c_colab.py prepare --source-mode snapshot --source-snapshot-dir /content/drive/MyDrive/OBAD/data/dataModel/l1_adaptation/source_snapshots/l1_candidate_c_source_current --adaptation-audit-dir /content/drive/MyDrive/OBAD/data/realtime_audit/l1_adaptation_eval_20260715_162915 --candidate-run-id l1_candidate_c_current --candidate-package-dir /content/drive/MyDrive/OBAD/data/dataModel/l1_adaptation/l1_candidate_c_current --resume

L1 CANDIDATE C PREPARATION FINISHED SUCCESSFULLY


In [6]:
#cell 7
from pathlib import Path
import json

state_path = PACKAGE / "run_state.json"

assert state_path.exists(), state_path

state = json.loads(
    state_path.read_text(encoding="utf-8")
)

print(json.dumps(state, indent=2, ensure_ascii=False))

machines = state.get("machines", {})

complete = {
    machine_id: status
    for machine_id, status in machines.items()
    if status == "COMPLETE"
}

not_complete = {
    machine_id: status
    for machine_id, status in machines.items()
    if status != "COMPLETE"
}

print("COMPLETE:", len(complete))
print("TOTAL:", len(machines))
print("NOT COMPLETE:", not_complete)

assert len(machines) == 14, machines
assert len(complete) == 14, not_complete
assert not not_complete, not_complete

{
  "run_id": "l1_candidate_c_current",
  "machines": {
    "11": "COMPLETE",
    "36": "COMPLETE",
    "37": "COMPLETE",
    "45": "COMPLETE",
    "46": "COMPLETE",
    "47": "COMPLETE",
    "48": "COMPLETE",
    "49": "COMPLETE",
    "50": "COMPLETE",
    "51": "COMPLETE",
    "56": "COMPLETE",
    "58": "COMPLETE",
    "59": "COMPLETE",
    "67": "COMPLETE"
  },
  "source_snapshot_hash": "bd01586c29a3854d1ec14d6f15edc19ff5aec08b2061b55a26349adfb15624cd"
}
COMPLETE: 14
TOTAL: 14
NOT COMPLETE: {}


In [7]:
#cell 8:
run_checked([
    sys.executable,
    "-u",
    "modeling/l1_tcn/scripts/run_candidate_c_colab.py",
    "validate-package",
    "--candidate-package-dir",
    str(PACKAGE),
])

CompletedProcess(args=['/usr/bin/python3', '-u', 'modeling/l1_tcn/scripts/run_candidate_c_colab.py', 'validate-package', '--candidate-package-dir', '/content/drive/MyDrive/OBAD/data/dataModel/l1_adaptation/l1_candidate_c_current'], returncode=0)

In [8]:
#cell 8,5
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

records = []

raw_root = SNAPSHOT / "fact"
canonical_root = PACKAGE / "canonical"

machine_dirs = sorted(
    raw_root.glob("machine_id=*"),
    key=lambda p: int(p.name.split("=")[1]),
)

for raw_dir in machine_dirs:
    machine_id = raw_dir.name.split("=")[1]

    raw_file = raw_dir / "events.parquet"
    canonical_file = (
        canonical_root
        / f"machine_id={machine_id}"
        / "events.parquet"
    )
    manifest_file = (
        canonical_root
        / f"machine_id={machine_id}"
        / "manifest.json"
    )

    raw_rows = pq.ParquetFile(raw_file).metadata.num_rows

    canonical_rows = (
        pq.ParquetFile(canonical_file).metadata.num_rows
        if canonical_file.exists()
        else 0
    )

    manifest = {}

    if manifest_file.exists():
        manifest = json.loads(
            manifest_file.read_text(encoding="utf-8")
        )

    dropped_rows = raw_rows - canonical_rows

    records.append({
        "machine_id": int(machine_id),
        "raw_rows": raw_rows,
        "canonical_rows": canonical_rows,
        "dropped_rows": dropped_rows,
        "retained_pct": round(
            canonical_rows / raw_rows * 100,
            2,
        ) if raw_rows else None,
        "manifest_row_fields": {
            key: value
            for key, value in manifest.items()
            if any(
                token in key.lower()
                for token in [
                    "row",
                    "drop",
                    "filter",
                    "exclude",
                    "invalid",
                ]
            )
        },
    })

result = pd.DataFrame(records)

display(
    result[
        [
            "machine_id",
            "raw_rows",
            "canonical_rows",
            "dropped_rows",
            "retained_pct",
        ]
    ]
)

print({
    "raw_total": int(result["raw_rows"].sum()),
    "canonical_total": int(result["canonical_rows"].sum()),
    "dropped_total": int(result["dropped_rows"].sum()),
    "retained_pct": round(
        result["canonical_rows"].sum()
        / result["raw_rows"].sum()
        * 100,
        2,
    ),
})

print("\n=== MANIFEST FILTER/ROW FIELDS ===")

for item in records:
    print(
        item["machine_id"],
        json.dumps(
            item["manifest_row_fields"],
            ensure_ascii=False,
            indent=2,
        ),
    )

,machine_id,raw_rows,canonical_rows,dropped_rows,retained_pct
0,11,187466,187465,1,100.00
1,36,181656,181655,1,100.00
2,37,115265,115264,1,100.00
3,45,213994,213993,1,100.00
4,46,63162,63161,1,100.00
5,47,45640,45639,1,100.00
6,48,83424,83423,1,100.00
7,49,12275,12274,1,99.99
8,50,148589,148588,1,100.00
9,51,16184,16183,1,99.99


{'raw_total': 1367105, 'canonical_total': 1367091, 'dropped_total': 14, 'retained_pct': np.float64(100.0)}

=== MANIFEST FILTER/ROW FIELDS ===
11 {
  "rows": 187465
}
36 {
  "rows": 181655
}
37 {
  "rows": 115264
}
45 {
  "rows": 213993
}
46 {
  "rows": 63161
}
47 {
  "rows": 45639
}
48 {
  "rows": 83423
}
49 {
  "rows": 12274
}
50 {
  "rows": 148588
}
51 {
  "rows": 16183
}
56 {
  "rows": 100325
}
58 {
  "rows": 43962
}
59 {
  "rows": 55610
}
67 {
  "rows": 99549
}


In [9]:
#cell 9:
summary_path = PACKAGE / "manifests/summary.json"

assert summary_path.exists(), summary_path

summary = json.loads(
    summary_path.read_text(encoding="utf-8")
)

print(json.dumps(summary, indent=2, ensure_ascii=False))

allowed_results = {
    "L1_CANDIDATE_C_PACKAGE_READY_FOR_COLAB_TRAINING",
    "FUTURE_LABEL_COVERAGE_INSUFFICIENT_BUT_PACKAGE_READY",
}

result = summary.get("result")

assert result in allowed_results, {
    "result": result,
    "allowed_results": sorted(allowed_results),
}

{
  "result": "L1_CANDIDATE_C_PACKAGE_READY_FOR_COLAB_TRAINING",
  "machine_complete": 14,
  "machine_total": 14,
  "raw_rows": 1367105,
  "canonical_rows": 1367091,
  "closed_rows": 1367091,
  "dropped_rows": 14,
  "errors": [],
  "refreshed": true
}


In [10]:
#cell 10:
run_checked([
    sys.executable,
    "modeling/l1_tcn/scripts/run_candidate_c_colab.py",
    "train",
    "--candidate-package-dir",
    str(PACKAGE),
    "--profile",
    "lenient",
    "--device",
    "cuda",
    "--resume",
])

CompletedProcess(args=['/usr/bin/python3', 'modeling/l1_tcn/scripts/run_candidate_c_colab.py', 'train', '--candidate-package-dir', '/content/drive/MyDrive/OBAD/data/dataModel/l1_adaptation/l1_candidate_c_current', '--profile', 'lenient', '--device', 'cuda', '--resume'], returncode=0)

In [ ]:
#chạy kiểm tra không tiến độ cell 10
'''from pathlib import Path
import subprocess
import time

PROJECT_ROOT = Path("/content/drive/MyDrive/OBAD")

ARTIFACT = (
    PROJECT_ROOT
    / "modeling/l1_tcn/artifacts_candidates"
    / "l1_candidate_c_current/current_only/lenient"
)

print("=== PROCESS ===")

result = subprocess.run(
    [
        "bash",
        "-lc",
        "ps -ef | grep -E 'run_candidate_c_colab|train.py' | grep -v grep || true",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout or "Không còn process train.")

print("\n=== GPU ===")

subprocess.run(["nvidia-smi"])

print("\n=== ARTIFACT FILES ===")

if ARTIFACT.exists():
    files = []

    for path in ARTIFACT.rglob("*"):
        if path.is_file():
            files.append((
                path.stat().st_mtime,
                path.stat().st_size,
                path,
            ))

    files.sort(reverse=True)

    for mtime, size, path in files[:30]:
        print({
            "age_seconds": round(time.time() - mtime, 1),
            "size": size,
            "path": str(path.relative_to(ARTIFACT)),
        })
else:
    print("Artifact directory chưa được tạo:", ARTIFACT)'''

In [11]:
#cell 11:
run_checked([sys.executable,'modeling/l1_tcn/scripts/run_candidate_c_colab.py','validate-artifact','--candidate-package-dir',str(PACKAGE),'--profile','lenient'])


CompletedProcess(args=['/usr/bin/python3', 'modeling/l1_tcn/scripts/run_candidate_c_colab.py', 'validate-artifact', '--candidate-package-dir', '/content/drive/MyDrive/OBAD/data/dataModel/l1_adaptation/l1_candidate_c_current', '--profile', 'lenient'], returncode=0)

In [12]:
#cell 12:
run_checked([sys.executable,'modeling/l1_tcn/scripts/run_candidate_c_colab.py','train','--candidate-package-dir',str(PACKAGE),'--profile','strict','--device','cuda','--resume'])


CompletedProcess(args=['/usr/bin/python3', 'modeling/l1_tcn/scripts/run_candidate_c_colab.py', 'train', '--candidate-package-dir', '/content/drive/MyDrive/OBAD/data/dataModel/l1_adaptation/l1_candidate_c_current', '--profile', 'strict', '--device', 'cuda', '--resume'], returncode=0)

In [13]:
#cell 13:
run_checked([sys.executable,'modeling/l1_tcn/scripts/run_candidate_c_colab.py','validate-artifact','--candidate-package-dir',str(PACKAGE),'--profile','strict'])


CompletedProcess(args=['/usr/bin/python3', 'modeling/l1_tcn/scripts/run_candidate_c_colab.py', 'validate-artifact', '--candidate-package-dir', '/content/drive/MyDrive/OBAD/data/dataModel/l1_adaptation/l1_candidate_c_current', '--profile', 'strict'], returncode=0)

In [14]:
#cell 14:
ARTIFACT=PROJECT_ROOT/'modeling/l1_tcn/artifacts_candidates'/RUN_ID/'current_only'
run_checked([sys.executable,'modeling/l1_tcn/scripts/run_candidate_c_colab.py','evaluate','--candidate-package-dir',str(PACKAGE),'--adaptation-audit-dir',str(ADAPTATION),'--candidate-artifact-dir',str(ARTIFACT)])


CompletedProcess(args=['/usr/bin/python3', 'modeling/l1_tcn/scripts/run_candidate_c_colab.py', 'evaluate', '--candidate-package-dir', '/content/drive/MyDrive/OBAD/data/dataModel/l1_adaptation/l1_candidate_c_current', '--adaptation-audit-dir', '/content/drive/MyDrive/OBAD/data/realtime_audit/l1_adaptation_eval_20260715_162915', '--candidate-artifact-dir', '/content/drive/MyDrive/OBAD/modeling/l1_tcn/artifacts_candidates/l1_candidate_c_current/current_only'], returncode=0)

In [ ]:
#cell chẩn đoán: Recovery / Diagnostics — chỉ chạy khi bị ngắt
from pathlib import Path
import json
import subprocess
import time

PROJECT_ROOT = Path("/content/drive/MyDrive/OBAD")

PACKAGE = (
    PROJECT_ROOT
    / "data/dataModel/l1_adaptation"
    / "l1_candidate_c_current"
)

print("=== RUNNING PROCESSES ===")

ps = subprocess.run(
    [
        "bash",
        "-lc",
        "ps -ef | grep -E 'score_new_events|run_candidate_c_colab' | grep -v grep || true",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(ps.stdout or "No Candidate C process found.")

print("\n=== PACKAGE EXISTS ===")
print(PACKAGE.exists(), PACKAGE)

print("\n=== RUN STATE ===")

state_candidates = [
    PACKAGE / "run_state.json",
    PACKAGE / "manifests/run_state.json",
]

state_found = False

for state_path in state_candidates:
    if state_path.exists():
        state_found = True
        print("State path:", state_path)
        try:
            state = json.loads(state_path.read_text(encoding="utf-8"))
            print(json.dumps(state, indent=2, ensure_ascii=False)[:20000])
        except Exception as exc:
            print("Could not read state:", repr(exc))

if not state_found:
    print("No run_state.json found.")

print("\n=== COMPLETED PARTITIONS ===")

success_files = list(PACKAGE.rglob("_SUCCESS")) if PACKAGE.exists() else []

print("SUCCESS count:", len(success_files))

for path in success_files[:100]:
    print(path.relative_to(PACKAGE))

print("\n=== RECENT FILES ===")

recent = []

if PACKAGE.exists():
    for path in PACKAGE.rglob("*"):
        try:
            if path.is_file():
                recent.append(
                    (
                        path.stat().st_mtime,
                        path.stat().st_size,
                        path,
                    )
                )
        except OSError:
            pass

recent.sort(reverse=True)

for mtime, size, path in recent[:40]:
    age_seconds = time.time() - mtime
    print(
        {
            "age_seconds": round(age_seconds, 1),
            "size": size,
            "path": str(path.relative_to(PACKAGE)),
        }
    )

In [ ]:
#cell đánh giá, báo cáo sau khi fineturn
from pathlib import Path
import json
import time

roots = {
    "PACKAGE": PACKAGE,
    "ARTIFACT": ARTIFACT,
    "ADAPTATION": ADAPTATION,
}

keywords = (
    "metric",
    "evaluation",
    "decision",
    "gate",
    "threshold",
    "contract",
    "summary",
    "history",
    "comparison",
    "candidate",
)

found = []

for root_name, root in roots.items():
    if not root.exists():
        print("MISSING ROOT:", root_name, root)
        continue

    for path in root.rglob("*.json"):
        relative = path.relative_to(root)
        searchable = str(relative).lower()

        if any(keyword in searchable for keyword in keywords):
            found.append((
                path.stat().st_mtime,
                root_name,
                path,
                relative,
            ))

found.sort(reverse=True)

print("=== REPORT FILES ===")

for mtime, root_name, path, relative in found:
    print({
        "root": root_name,
        "path": str(relative),
        "size": path.stat().st_size,
        "age_seconds": round(time.time() - mtime, 1),
    })

print("\n=== REPORT CONTENT ===")

for _, root_name, path, relative in found:
    print(f"\n\n===== {root_name}/{relative} =====")

    try:
        content = json.loads(path.read_text(encoding="utf-8"))
        print(json.dumps(content, indent=2, ensure_ascii=False)[:30000])
    except Exception as exc:
        print("READ ERROR:", repr(exc))